# SEC Hyperscaler + Secondaries Scanner — token-free (Colab)

Pulls **real SEC EDGAR data** (XBRL structured facts + full-text search) directly from the source — no key, no
token, no LLM calls. Built to operationalize the vault's fragility discipline: instead of waiting for a WSJ
piece to surface a footnote (the Google/SpaceX $94.1B stake — the paper had to do that for us on 2026-07-23),
this pulls the same filings on demand and flags the things the vault has been manually digging for all week.

**Three buckets, matching the quality-ladder razor:**
- **HYPERSCALERS** (cash-rich core): GOOGL, MSFT, AMZN, META
- **SECONDARIES** (chip/memory "sellers"): NVDA, AMD, INTC, MU, AVGO, TSM
- **PERIPHERY** (levered/neocloud, context only): ORCL, CRWV

**What it checks, per the vault's own threads:**
1. **The depreciation-schedule test** (Dowd/Goldman claim, 2026-07-24) — implied useful life = PP&E ÷ annualized
   depreciation. Lengthening while capex accelerates = the "artificial earnings" flag, made falsifiable.
2. **The off-balance-sheet commitments tracker** (the "$811B" / Beignet thread) — `LongTermPurchaseCommitmentAmount`
   and lease liabilities, trended over quarters.
3. **The unrealized-equity-gains catcher** (the SpaceX-stake mechanism) — `EquitySecuritiesFvNiGainLoss`, so the
   next one shows up here before a reporter has to find it.
4. **The FCF-proxy** (the "Google's first negative-cash-flow quarter" Facebook-chart thread) — operating cash
   flow minus capex.
5. **Secondaries fundamentals** — revenue, gross margin, inventory trend (memory/chip cycle read).
6. **Ad-hoc full-text search** — the Schedule-D-style "go find the receipt" tool: search any keyword (a
   counterparty name, "special purpose entity", "guarantee", "take-or-pay") across recent filings.

Run cells top to bottom. Every network call is wrapped defensively — a missing tag prints `N/A`, it never
crashes the notebook. Re-run any single cell any time; nothing is cached beyond the session.


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
try:
    import pandas as pd, requests
except Exception:
    _pip('pandas'); _pip('requests')
    import pandas as pd, requests
import time, json
from datetime import datetime

# SEC requires a descriptive User-Agent (name + contact) on every request, or it 403s/429s.
# Edit the email below if you want — SEC doesn't validate it, but keep the format (name + contact).
HEADERS = {'User-Agent': 'INMA-Research-Vault research-contact@example.com'}
SLEEP = 0.15  # stay well under SEC's ~10 req/sec limit across all the calls this notebook makes

def _get(url, params=None, timeout=15):
    """Defensive GET: never raises, returns None on any failure."""
    try:
        r = requests.get(url, headers=HEADERS, params=params, timeout=timeout)
        time.sleep(SLEEP)
        if r.status_code != 200:
            return None
        return r.json()
    except Exception:
        return None

print('Fetching SEC ticker->CIK map (one-time, ~10,400 companies)...')
_TICKER_MAP = _get('https://www.sec.gov/files/company_tickers.json')
if _TICKER_MAP is None:
    print('⚠️ Could not reach SEC — check internet connection / try re-running this cell.')
    TICKER_TO_CIK = {}
else:
    TICKER_TO_CIK = {v['ticker'].upper(): str(v['cik_str']).zfill(10) for v in _TICKER_MAP.values()}
    print(f'  Loaded {len(TICKER_TO_CIK)} tickers.')

def cik_for(ticker):
    return TICKER_TO_CIK.get(ticker.upper())


## Watchlist — the three buckets

In [ ]:
HYPERSCALERS = ['GOOGL', 'MSFT', 'AMZN', 'META']
SECONDARIES  = ['NVDA', 'AMD', 'INTC', 'MU', 'AVGO', 'TSM']
PERIPHERY    = ['ORCL', 'CRWV']  # levered/neocloud — context, not core

ALL_TICKERS = HYPERSCALERS + SECONDARIES + PERIPHERY

print('Resolving CIKs...')
for t in ALL_TICKERS:
    c = cik_for(t)
    print(f'  {t:6s} -> CIK {c}' if c else f'  {t:6s} -> NOT FOUND (ticker may differ on EDGAR, e.g. class shares)')


## XBRL concept puller

`companyconcept` returns the full historical time series for ONE tag. Tag names aren't perfectly standardized
across companies/years, so each concept below tries a short list of candidate tag names and uses the first
that returns data.

In [ ]:
def get_concept(cik, tag, taxonomy='us-gaap'):
    """Pull one XBRL concept's full history for a company. Returns a DataFrame or None."""
    if cik is None:
        return None
    d = _get(f'https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/{taxonomy}/{tag}.json')
    if d is None or 'units' not in d:
        return None
    units = d['units']
    usd = units.get('USD')
    if usd is None:   # IFRS filers sometimes report in other currencies; take the first unit available
        usd = next(iter(units.values()), None)
    if not usd:
        return None
    df = pd.DataFrame(usd)
    if df.empty:
        return None
    df['end'] = pd.to_datetime(df['end'])
    df = df.sort_values('end')
    return df

def get_concept_any(cik, candidate_tags, taxonomy='us-gaap'):
    """Try each candidate tag in order; return (tag_used, DataFrame) for the first that has data."""
    for tag in candidate_tags:
        df = get_concept(cik, tag, taxonomy)
        if df is not None and len(df) > 0:
            return tag, df
    return None, None

def latest_annualized(df, period_days_min=80):
    """For a flow (income-statement/cash-flow) concept, get the most recent quarterly-ish value, annualized.
    Filters to entries that look like a single quarter (roughly 80-100 days) to avoid mixing YTD and Q figures."""
    if df is None or 'start' not in df.columns:
        return None
    d2 = df.dropna(subset=['start']).copy()
    d2['start'] = pd.to_datetime(d2['start'])
    d2['days'] = (d2['end'] - d2['start']).dt.days
    q = d2[(d2['days'] >= period_days_min) & (d2['days'] <= 100)]
    if q.empty:
        return None
    row = q.sort_values('end').iloc[-1]
    return row['val'] * 4, row['end']

def get_concept_max(cik, candidate_tags, taxonomy='us-gaap'):
    """For concepts where companies tag PARTIAL scopes under some element names (revenue is the
    classic case — a 'contract revenue' tag can match a SEGMENT slice while GrossProfit is
    company-wide, which is how a 1969% gross margin happened on NVDA in testing):
    try ALL candidates and keep the one whose latest annualized value is LARGEST.
    Total >= any slice, so max = the company-wide figure."""
    best = (None, None, None)   # tag, df, annualized tuple
    for tag in candidate_tags:
        df = get_concept(cik, tag, taxonomy)
        if df is None or len(df) == 0:
            continue
        ann = latest_annualized(df)
        if ann is None:
            continue
        if best[2] is None or ann[0] > best[2][0]:
            best = (tag, df, ann)
    return best

# Candidate tag lists per concept (order = preference)
TAGS_PPE          = ['PropertyPlantAndEquipmentNet']
TAGS_DEPRECIATION = ['Depreciation', 'DepreciationDepletionAndAmortization', 'DepreciationAmortizationAndAccretionNet']
# AMZN tags capex under PaymentsToAcquireProductiveAssets — caught in testing when its capex printed
# $7.4B annualized (an order of magnitude low) and silently corrupted the FCF proxy. Order matters less
# here than coverage: the snapshot code takes the LARGEST annualized value across candidates for capex too.
TAGS_CAPEX        = ['PaymentsToAcquirePropertyPlantAndEquipment', 'PaymentsToAcquireProductiveAssets', 'PaymentsForCapitalImprovements']
TAGS_OCF          = ['NetCashProvidedByUsedInOperatingActivities', 'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations']
TAGS_EQUITY_GAIN  = ['EquitySecuritiesFvNiGainLoss', 'MarketableSecuritiesGainLoss', 'UnrealizedGainLossOnInvestments']
TAGS_PURCHASE_OBL = ['LongTermPurchaseCommitmentAmount', 'UnconditionalPurchaseObligationBalanceSheetAmount', 'PurchaseObligation']
TAGS_LEASE_LIAB   = ['OperatingLeaseLiabilityNoncurrent']
TAGS_REVENUE      = ['RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues', 'RevenueFromContractWithCustomerIncludingAssessedTax']
TAGS_GROSS_PROFIT = ['GrossProfit']
TAGS_INVENTORY    = ['InventoryNet']

print('Helpers ready.')


## Hyperscaler earnings-quality dashboard

For each hyperscaler: PP&E, implied useful-life trend (the depreciation test), capex run-rate, FCF proxy,
latest unrealized equity-securities gain/loss (the SpaceX-stake mechanism), and off-balance-sheet purchase
commitments. Everything trended so you can see direction, not just a snapshot.

In [ ]:
def hyperscaler_snapshot(ticker):
    cik = cik_for(ticker)
    if cik is None:
        return {'ticker': ticker, 'error': 'CIK not found'}

    out = {'ticker': ticker}

    # PP&E + depreciation -> implied useful life (years). Lengthening = the earnings-quality flag.
    # KNOWN QUIRK (confirmed live on GOOGL and META): companies sometimes stop tagging a concept with
    # the same XBRL element (e.g. switch to a disaggregated PP&E breakdown) — the tag goes quietly
    # stale even though the company keeps filing. If ppe_as_of and dep_as_of diverge by more than
    # ~120 days, the ratio is comparing a STALE balance to a FRESH flow — flagged explicitly.
    _, ppe = get_concept_any(cik, TAGS_PPE)
    dep_tag, dep = get_concept_any(cik, TAGS_DEPRECIATION)
    if ppe is not None and dep is not None:
        dep_ann = latest_annualized(dep)
        ppe_latest = ppe.sort_values('end').iloc[-1]
        if dep_ann is not None and dep_ann[0]:
            out['ppe_net_latest_$B'] = round(ppe_latest['val'] / 1e9, 1)
            out['ppe_as_of'] = ppe_latest['end'].date().isoformat()
            out['dep_as_of'] = dep_ann[1].date().isoformat()
            staleness_days = abs((dep_ann[1] - ppe_latest['end']).days)
            useful_life = round(ppe_latest['val'] / dep_ann[0], 1)
            if staleness_days > 120:
                out['implied_useful_life_yrs'] = f"{useful_life} ⚠️ STALE PP&E TAG ({staleness_days}d gap vs dep — verify in the 10-Q text, don't trust this ratio as-is)"
            else:
                out['implied_useful_life_yrs'] = useful_life
        else:
            out['implied_useful_life_yrs'] = 'N/A (depreciation not annualizable)'
    else:
        out['implied_useful_life_yrs'] = f'N/A (dep tag: {dep_tag or "none found"})'

    # Capex run-rate (annualized). Uses the LARGEST value across candidate tags — AMZN tags its capex
    # under PaymentsToAcquireProductiveAssets, and first-match logic printed $7.4B (10x low) in testing,
    # which silently corrupted the FCF proxy. Max-across-candidates is robust to that.
    capex_tag, _, capex_ann = get_concept_max(cik, TAGS_CAPEX)
    out['capex_annualized_$B'] = round(capex_ann[0] / 1e9, 1) if capex_ann else 'N/A'
    if capex_tag:
        out['capex_tag'] = capex_tag

    # FCF proxy = OCF - capex (both annualized off their latest quarters)
    _, ocf = get_concept_any(cik, TAGS_OCF)
    ocf_ann = latest_annualized(ocf) if ocf is not None else None
    if ocf_ann and capex_ann:
        out['fcf_proxy_annualized_$B'] = round((ocf_ann[0] - capex_ann[0]) / 1e9, 1)
    else:
        out['fcf_proxy_annualized_$B'] = 'N/A'

    # Unrealized equity-securities gain/loss — the SpaceX-stake catcher. Most recent single value, not annualized.
    eq_tag, eq = get_concept_any(cik, TAGS_EQUITY_GAIN)
    if eq is not None:
        row = eq.sort_values('end').iloc[-1]
        out['latest_equity_gain_$B'] = round(row['val'] / 1e9, 2)
        out['equity_gain_period_end'] = row['end'].date().isoformat()
        out['equity_gain_tag'] = eq_tag
    else:
        out['latest_equity_gain_$B'] = 'N/A (not tagged)'

    # Off-balance-sheet purchase commitments — the $811B/Beignet thread
    po_tag, po = get_concept_any(cik, TAGS_PURCHASE_OBL)
    if po is not None:
        row = po.sort_values('end').iloc[-1]
        out['purchase_commitments_$B'] = round(row['val'] / 1e9, 1)
        out['purchase_commitments_tag'] = po_tag
    else:
        out['purchase_commitments_$B'] = 'N/A (not tagged — check 10-K text/full-text search)'

    return out

print('Pulling hyperscaler snapshots (this hits several SEC endpoints per ticker, ~10-20s total)...\n')
hyperscaler_rows = []
for t in HYPERSCALERS:
    snap = hyperscaler_snapshot(t)
    hyperscaler_rows.append(snap)
    print(f"--- {t} ---")
    for k, v in snap.items():
        if k != 'ticker':
            print(f'  {k:28s}: {v}')
    print()

hyperscaler_df = pd.DataFrame(hyperscaler_rows)


## Secondaries fundamentals — revenue, margin, inventory (the memory/chip-cycle read)

In [ ]:
# Foreign private issuers file 20-F under IFRS — us-gaap tags don't exist for them.
IFRS_FILERS = {'TSM'}

def secondary_snapshot(ticker):
    cik = cik_for(ticker)
    if cik is None:
        return {'ticker': ticker, 'error': 'CIK not found'}
    out = {'ticker': ticker}

    if ticker.upper() in IFRS_FILERS:
        out['note'] = 'Foreign private issuer (20-F, IFRS taxonomy) — us-gaap tags do not apply. Use the full-text search cell or read the 20-F directly.'
        return out

    # Revenue: take the LARGEST annualized value across candidate tags. First-match logic burned us in
    # testing — NVDA's contract-revenue tag matched a partial slice ($12.4B "revenue") while GrossProfit
    # was company-wide, printing a 1969% gross margin. Total >= any slice, so max = company-wide.
    rev_tag, _, rev_ann = get_concept_max(cik, TAGS_REVENUE)
    out['revenue_annualized_$B'] = round(rev_ann[0] / 1e9, 1) if rev_ann else 'N/A'
    if rev_tag:
        out['revenue_tag'] = rev_tag

    _, gp = get_concept_any(cik, TAGS_GROSS_PROFIT)
    gp_ann = latest_annualized(gp) if gp is not None else None
    if gp_ann and rev_ann and rev_ann[0]:
        gm = round(100 * gp_ann[0] / rev_ann[0], 1)
        # Sanity guard: >95% or <0% gross margin on a hardware name = the tags are almost certainly
        # measuring different scopes or periods. Print the number but refuse to let it pass as clean.
        if gm > 95 or gm < 0:
            out['gross_margin_pct'] = f'{gm} ⚠️ SCOPE MISMATCH LIKELY — rev tag ({rev_tag}) and GrossProfit may cover different scopes/periods; verify in the filing before quoting'
        else:
            out['gross_margin_pct'] = gm
    else:
        out['gross_margin_pct'] = 'N/A'

    _, inv = get_concept_any(cik, TAGS_INVENTORY)
    if inv is not None and len(inv) >= 2:
        inv_sorted = inv.sort_values('end')
        latest = inv_sorted.iloc[-1]
        prior_yr = inv_sorted[inv_sorted['end'] <= latest['end'] - pd.Timedelta(days=330)]
        out['inventory_latest_$B'] = round(latest['val'] / 1e9, 2)
        out['inventory_as_of'] = latest['end'].date().isoformat()
        if not prior_yr.empty:
            yoy = 100 * (latest['val'] / prior_yr.iloc[-1]['val'] - 1)
            out['inventory_yoy_pct'] = round(yoy, 1)
        else:
            out['inventory_yoy_pct'] = 'N/A (no year-ago comp)'
    else:
        out['inventory_latest_$B'] = 'N/A'

    return out

print('Pulling secondaries snapshots...\n')
secondary_rows = []
for t in SECONDARIES + PERIPHERY:
    snap = secondary_snapshot(t)
    secondary_rows.append(snap)
    print(f"--- {t} ---")
    for k, v in snap.items():
        if k != 'ticker':
            print(f'  {k:22s}: {v}')
    print()

secondary_df = pd.DataFrame(secondary_rows)


## Ad-hoc full-text search — the "go find the receipt" tool

Search any keyword across recent SEC filings (10-K/10-Q/8-K, all companies or restricted to the watchlist).
This is the Schedule-D move: instead of waiting for a reporter to surface a footnote, search for it — a
counterparty name (`"CoreWeave"`, `"OpenAI"`, `"SpaceX"`), a structure (`"special purpose entity"`,
`"take-or-pay"`, `"guarantee"`), or anything else you're trying to verify.

In [ ]:
def search_edgar(keyword, forms='10-K,10-Q,8-K', start='2025-01-01', end=None, restrict_to_watchlist=True, max_results=15):
    """Full-text search across SEC EDGAR filings. Returns a DataFrame of hits."""
    if end is None:
        end = datetime.today().strftime('%Y-%m-%d')
    params = {
        'q': f'"{keyword}"',
        'forms': forms,
        'dateRange': 'custom',
        'startdt': start,
        'enddt': end,
    }
    d = _get('https://efts.sec.gov/LATEST/search-index', params=params)
    if d is None:
        print('⚠️ Search failed — check connection and try again.')
        return pd.DataFrame()

    total = d.get('hits', {}).get('total', {}).get('value', 0)
    hits = d.get('hits', {}).get('hits', [])
    rows = []
    watch_ciks = {cik_for(t) for t in ALL_TICKERS if cik_for(t)}
    for h in hits:
        src = h.get('_source', {})
        cik_list = src.get('ciks', [])
        hit_cik = cik_list[0].zfill(10) if cik_list else None
        if restrict_to_watchlist and hit_cik not in watch_ciks:
            continue
        rows.append({
            'company': ', '.join(src.get('display_names', [])),
            'form': ', '.join(src.get('forms', [])) if isinstance(src.get('forms'), list) else src.get('forms'),
            'filed': src.get('file_date'),
            'accession': h.get('_id'),
        })
        if len(rows) >= max_results:
            break
    print(f'Total hits for "{keyword}": {total} (showing {"watchlist-only" if restrict_to_watchlist else "all companies"}, up to {max_results})')
    return pd.DataFrame(rows)

# Example — try your own keywords by editing this cell:
example_results = search_edgar('SpaceX', restrict_to_watchlist=True)
example_results


## One-look summary — RUN THIS LAST (prints + saves to Drive)

This is the results cell. It combines both dashboards into one report, prints the compact version, and
**saves three files** — two CSVs and a markdown summary — so the run is a dated artifact, not a scrollback.

- **In Colab:** it mounts Google Drive (you'll get a one-tap auth prompt) and writes to
  `MyDrive/vault_scans/`. Each run is stamped with the date, so you build a history you can diff.
- **Anywhere else / Drive declined:** it falls back to the local working directory and tells you where.

The markdown file is paste-ready for the vault (`raw/` drop zone) — it's formatted as a DATA block with
source + date, per the firewall.

In [ ]:
import os

# ---------- 1. Build the summary tables ----------
hs_cols = ['ticker', 'implied_useful_life_yrs', 'capex_annualized_$B', 'fcf_proxy_annualized_$B',
           'latest_equity_gain_$B', 'equity_gain_period_end', 'purchase_commitments_$B',
           'ppe_net_latest_$B', 'ppe_as_of']
sec_cols = ['ticker', 'revenue_annualized_$B', 'gross_margin_pct', 'inventory_latest_$B', 'inventory_yoy_pct']

hs_view  = hyperscaler_df[[c for c in hs_cols  if c in hyperscaler_df.columns]]
sec_view = secondary_df[[c for c in sec_cols if c in secondary_df.columns]]

run_stamp = datetime.today().strftime('%Y-%m-%d_%H%M')

# ---------- 2. Auto-flags: surface the loudest signals so the summary reads itself ----------
flags = []
for _, r in hyperscaler_df.iterrows():
    v = r.get('latest_equity_gain_$B')
    if isinstance(v, (int, float)) and abs(v) >= 5:
        flags.append(f"{r['ticker']}: unrealized equity-securities gain/loss of ${v}B "
                     f"(period end {r.get('equity_gain_period_end', '?')}) — the SpaceX-stake mechanism; "
                     f"check what is being marked and whether the credit market agrees with the mark.")
    v = r.get('fcf_proxy_annualized_$B')
    if isinstance(v, (int, float)) and v < 0:
        flags.append(f"{r['ticker']}: NEGATIVE annualized FCF proxy (${v}B) — capex outrunning operating cash.")
    v = r.get('purchase_commitments_$B')
    if isinstance(v, (int, float)) and v >= 100:
        flags.append(f"{r['ticker']}: ${v}B in off-balance-sheet purchase commitments — the footnote leverage.")
    ul = str(r.get('implied_useful_life_yrs', ''))
    if 'STALE' in ul:
        flags.append(f"{r['ticker']}: PP&E tag is stale ({ul.split('⚠️')[0].strip()} yrs shown) — "
                     f"useful-life ratio unreliable, verify in the 10-Q text.")
for _, r in secondary_df.iterrows():
    v = r.get('inventory_yoy_pct')
    if isinstance(v, (int, float)) and v >= 30:
        flags.append(f"{r['ticker']}: inventory +{v}% YoY — building faster than sales? (cycle-top tell if margins roll).")

# ---------- 3. Print the compact version ----------
print('=' * 72)
print(f'SEC SCANNER SUMMARY — run {run_stamp} (source: SEC EDGAR XBRL, as-filed)')
print('=' * 72)
print('\nHYPERSCALERS — earnings quality:')
print(hs_view.to_string(index=False))
print('\nSECONDARIES + PERIPHERY — fundamentals:')
print(sec_view.to_string(index=False))
print('\nAUTO-FLAGS (' + str(len(flags)) + '):')
if flags:
    for f in flags:
        print('  • ' + f)
else:
    print('  (none tripped this run)')

# ---------- 4. Save: Drive if available, local otherwise ----------
save_dir = '.'
try:
    from google.colab import drive as _gdrive   # only exists in Colab
    _gdrive.mount('/content/drive', force_remount=False)
    save_dir = '/content/drive/MyDrive/vault_scans'
    os.makedirs(save_dir, exist_ok=True)
except Exception:
    save_dir = os.getcwd()   # not Colab, or Drive declined — save locally

hs_path  = os.path.join(save_dir, f'sec_scan_hyperscalers_{run_stamp}.csv')
sec_path = os.path.join(save_dir, f'sec_scan_secondaries_{run_stamp}.csv')
md_path  = os.path.join(save_dir, f'sec_scan_summary_{run_stamp}.md')

hyperscaler_df.to_csv(hs_path, index=False)
secondary_df.to_csv(sec_path, index=False)

try:
    hs_md, sec_md = hs_view.to_markdown(index=False), sec_view.to_markdown(index=False)
except Exception:   # to_markdown needs 'tabulate'; fall back to plain text if missing
    hs_md, sec_md = hs_view.to_string(index=False), sec_view.to_string(index=False)

md_lines = [
    f'# SEC scanner summary — {run_stamp}',
    '',
    'Source: SEC EDGAR XBRL companyconcept API (as-filed figures, pulled directly — no intermediary).',
    'DATA only — no interpretation. Firewall: write any THESIS separately in the vault.',
    '',
    '## Hyperscalers — earnings quality',
    '',
    hs_md,
    '',
    '## Secondaries + periphery — fundamentals',
    '',
    sec_md,
    '',
    f'## Auto-flags ({len(flags)})',
    '',
]
md_lines += ['- ' + f for f in flags] if flags else ['- (none tripped this run)']
md_lines += [
    '',
    'Notes: "N/A (not tagged)" = the company does not report that concept under the standard XBRL tag;',
    'it is NOT evidence of absence — use the full-text search cell to check the actual filing text.',
    'Flow figures are latest-quarter annualized (x4). STALE-flagged ratios should not be quoted.',
]
with open(md_path, 'w') as fh:
    fh.write('\n'.join(md_lines))

print('\n' + '=' * 72)
print('SAVED:')
print('  ' + hs_path)
print('  ' + sec_path)
print('  ' + md_path + '   <-- paste-ready for the vault (raw/ drop zone)')
if save_dir == os.getcwd():
    print('\n  (Drive not available — saved to the local working directory instead.')
    print('   In Colab, files here vanish when the runtime recycles: download via the folder icon, left sidebar.)')
